In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.metrics import f1_score
import joblib
from pathlib import Path

In [2]:
DATA_DIR = Path("data/preprocessed")
SPLIT_DIR = DATA_DIR / "tokenized_splits"

In [ ]:
vectorizer = TfidfVectorizer(
    token_pattern=r"\d+",
    ngram_range=(1, 3),
    max_features=5000000
)

# relieable = ["reliable", "political", "clickbait", "unreliable"]
# # unrelieable_score = {
# #     'fake' : 1.0,
# #     'conspiracy' : 1.0,
# #     'rumor' : 0.9,
# #     'hate' : 0.9,
# #     'satire' : 0.9,
# #     'junksci' : 0.85,
# #     'bias' : 0.8,
# #     'unreliable' : 0.35,
# #     'clickbait' : 0.15,
# #     'political' : 0.05,
# #     'reliable' : 0.0,
# # }
# def type_to_value(t : str):
#     return 0 if t in relieable else 1

# def extract_df(df : pd.DataFrame, is_train : bool = False):
#     print("Categorizing type...")
#     y = df['type'].map(type_to_value)

#     print("Vectorizing content tokens...")
#     if is_train: X_content = vectorizer.fit_transform(df["tokens"])
#     else: X_content = vectorizer.transform(df["tokens"])

#     return X_content, y
def extract_df(split : str, is_train : bool = False):
    print(f"Reading {split}...")
    df = pd.read_csv(SPLIT_DIR / f"{split}.csv")

    print("Vectorizing content...")
    if is_train: X_content = vectorizer.fit_transform(df["tokens"])
    else: X_content = vectorizer.transform(df["tokens"])

    return X_content, df['is_reliable']

In [4]:
X_train, y_train = extract_df("train", is_train=True)

X_val, y_val = extract_df("val")

Reading train...
Vectorizing content...
Reading val...
Vectorizing content...


In [5]:
from sklearn.linear_model import LogisticRegression

logre_model = LogisticRegression(max_iter=500, C=10, class_weight='balanced', solver='sag')
logre_model.fit(X_train, y_train)

print(f1_score(y_val, logre_model.predict(X_val)))

# 805 -> 823 -> 838 -> 847 ->rel 872 -> 879 -> 901

0.8979530289893316


In [6]:
print(f1_score(y_train, logre_model.predict(X_train)))

0.9892691501801031


In [7]:
# from sklearn.svm import LinearSVC

# svm_model = LinearSVC(penalty='l1', C=1, class_weight='balanced')
# svm_model.fit(X_train, y_train)

# print(f"f1 score: {f1_score(y_val, svm_model.predict(X_val))}")

# 832 -> 841 -> 854 -> 880

In [8]:
# joblib.dump(logre_model, "data/models/logistic_regression.joblib")
# joblib.dump(vectorizer, "data/models/logistic_regression_vectorizer.joblib")